In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(
f"""
CREATE OR REPLACE TEMP VIEW diagnosis_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(Sequence AS INT) AS Sequence,
  CAST(DiagCode AS STRING) AS DiagCode,
  CAST(DiagDesc AS STRING) AS DiagDesc,
  CAST(Version AS INT) AS Version,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
WITH 
  obd AS (
    SELECT
        office_key,
        client_key,
        invoice_number,
        date_entered_key
    FROM {source_table}
    WHERE account_balance <> 0
    AND date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  office_data AS (
      SELECT
          o.date_entered_key,
          o.client_key,
          ofc.OfficeNumber,
          o.invoice_number
      FROM obd o
      LEFT JOIN {office_table} ofc
          ON o.office_key = ofc.OfficeKey
  ),
  client_diag AS (
      SELECT
          ClientKey,
          SourceSystemId,
          PrimaryDiagnosisCode
      FROM {client_table}
  ),
  diagnosis_cte AS (
    SELECT
      to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
      a.OfficeNumber AS FacilityCode,
      CASE
          WHEN UPPER(a.invoice_number) = 'ADV'
              THEN concat('ADV', ' - ', b.SourceSystemId)
          ELSE a.invoice_number
      END AS AcctNbr,
      1 AS Sequence,
      CASE
          WHEN b.PrimaryDiagnosisCode LIKE '%|%'
              THEN replace(b.PrimaryDiagnosisCode, '|', '')
          ELSE b.PrimaryDiagnosisCode
      END AS DiagCode,
      '' AS DiagDesc,
      10 AS Version,
      0 AS SourceSystemKey
    FROM office_data a
    LEFT JOIN client_diag b
        ON a.client_key = b.ClientKey
  ),
  diagnosis_clean AS (
    SELECT *,
    ROW_NUMBER() OVER (
          PARTITION BY
            AcctNbr, FacilityCode
          ORDER BY AcctNbr, DiagCode
      ) AS rn
    FROM diagnosis_cte
  )
  SELECT 
  ReportingDate, FacilityCode, AcctNbr, Sequence, DiagCode, DiagDesc, Version, SourceSystemKey
  FROM diagnosis_clean
  WHERE rn=1
) AS src
"""
)
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING diagnosis_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Sequence = src.Sequence,
    tgt.DiagCode = src.DiagCode,
    tgt.DiagDesc = src.DiagDesc,
    tgt.Version = src.Version,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Sequence,
    DiagCode,
    DiagDesc,
    Version,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Sequence,
    src.DiagCode,
    src.DiagDesc,
    src.Version,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)